<a href="https://colab.research.google.com/github/likhitnayak17-dev/AI_lab/blob/main/resolution_in_fol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

# Split literals in a clause
def split_literals(s):
    return re.findall(r'~?\s*[A-Za-z]\w*\([^)]*\)', s)


# Parse a literal
def parse_literal(lit):
    lit = lit.strip()
    neg = lit.startswith("~")
    if neg:
        lit = lit[1:].strip()

    pred, args = re.match(r'(\w+)\((.*?)\)', lit).groups()
    args = tuple(x.strip() for x in args.split(",") if x.strip())

    return neg, pred, args


# Convert literal into string
def make_literal(neg, pred, args):
    return ("~" if neg else "") + pred + "(" + ",".join(args) + ")"


# Check whether a term is a variable
def is_variable(x):
    return x[0].islower()


# Apply substitution
def apply_substitution(term, subst):
    while term in subst:
        term = subst[term]
    return term


# Unification algorithm
def unify(args1, args2, subst):
    for a, b in zip(args1, args2):
        a = apply_substitution(a, subst)
        b = apply_substitution(b, subst)

        if a == b:
            continue

        if is_variable(a):
            subst[a] = b
        elif is_variable(b):
            subst[b] = a
        else:
            return None

    return subst


# Rename variables to avoid conflicts
def rename_clause(clause, suffix):
    result = []
    for lit in clause:
        neg, pred, args = parse_literal(lit)
        new_args = [
            arg + suffix if is_variable(arg) else arg
            for arg in args
        ]
        result.append(make_literal(neg, pred, new_args))
    return result


# Apply substitution to a clause
def substitute_clause(clause, subst):
    result = []

    for lit in clause:
        neg, pred, args = parse_literal(lit)
        new_args = tuple(
            apply_substitution(arg, subst) for arg in args
        )
        result.append(make_literal(neg, pred, new_args))

    return result


# Check for tautology
def is_tautology(clause):
    literals = set(clause)

    for lit in clause:
        neg, pred, args = parse_literal(lit)
        opposite = make_literal(not neg, pred, args)

        if opposite in literals:
            return True

    return False


# Resolution between two clauses
def resolve(c1, c2):
    resolvents = []

    for i, l1 in enumerate(c1):
        for j, l2 in enumerate(c2):

            n1, p1, a1 = parse_literal(l1)
            n2, p2, a2 = parse_literal(l2)

            if n1 != n2 and p1 == p2 and len(a1) == len(a2):

                subst = unify(a1, a2, {})

                if subst is not None:
                    new_clause = (
                        c1[:i] + c1[i+1:] +
                        c2[:j] + c2[j+1:]
                    )

                    new_clause = substitute_clause(
                        new_clause, subst
                    )

                    new_clause = list(set(new_clause))

                    if not is_tautology(new_clause):
                        resolvents.append(new_clause)

    return resolvents


# Main resolution algorithm
def resolution(clauses):

    known = [list(set(c)) for c in clauses]
    count = 0

    while True:
        new_clauses = []

        for i in range(len(known)):
            for j in range(i + 1, len(known)):

                c1 = rename_clause(known[i], "x")
                c2 = rename_clause(known[j], "y")

                results = resolve(c1, c2)

                for res in results:
                    if len(res) == 0:
                        print("\nEmpty clause obtained!")
                        print("Contradiction found.")
                        return True

                    if res not in known and res not in new_clauses:
                        new_clauses.append(res)

        if not new_clauses:
            print("\nNo contradiction found.")
            return False

        known.extend(new_clauses)
        count += 1

        print("\nIteration", count)
        for clause in new_clauses:
            print(" OR ".join(clause))


# Taking input from user
clauses = []

n = int(input("Enter the number of clauses: "))

print("\nEnter clauses in CNF format.")
print("Example: P(x), ~Q(x)")

for i in range(n):
    s = input(f"Enter clause {i+1}: ")
    literals = split_literals(s)
    clauses.append(literals)

# Perform resolution
resolution(clauses)


Enter the number of clauses: 2

Enter clauses in CNF format.
Example: P(x), ~Q(x)
Enter clause 1: P(X)
Enter clause 2: Q(a)

No contradiction found.


False